In [ ]:
import requests
import json
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

def get_uplinks(api_key, application_id, device_id=None, limit=10, after="2020-08-20T00:00:00Z", field_mask=None):
    base_url = f"https://eu1.cloud.thethings.network/api/v3/as/applications/{application_id}/packages/storage/uplink_message"
    # If retrieving uplinks for a specific device, modify the URL
    if device_id:
        base_url = f"https://eu1.cloud.thethings.network/api/v3/as/applications/{application_id}/devices/{device_id}/packages/storage/uplink_message"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/json" 
    }
    params = {
        "limit": limit,
        "after": after
    }
    if field_mask:
        params["field_mask"] = field_mask
    response = requests.get(base_url, headers=headers, params=params)
    if response.status_code != 200:
        raise Exception(f"Error {response.status_code}: {response.text}")
    response = response.content.decode("utf-8")
    objs = []
    for line in response.split("\n"):
        if line.strip():
            objs.append(json.loads(line))
    return objs

In [ ]:
api_key = ""
application_id = ""
device_id = ""
limit = 0
after = "2020-10-10T10:10:10Z"
field_mask = "up.uplink_message.decoded_payload" 
uplink_data = get_uplinks(api_key, application_id, device_id, limit, after, field_mask)
data = []
for d in uplink_data:
    try:
        dd = d["result"]["uplink_message"]["decoded_payload"]
        dd["time"] = d["result"]["uplink_message"]["received_at"]
        data.append(dd)
    except:
        pass
# Stampa i dati in formato JSON
print(json.dumps(data, indent=2))
df = pd.DataFrame(data)
df["time"] = pd.to_datetime(df["time"])
df.sort_values("time", inplace=True)

display(df)

In [ ]:
df.rename(columns={"temperature_4": "Temperature (°C)",
                   "analog_in_8": "pH", 
                   "barometric_pressure_7": "Pressure (hPa)", 
                   "relative_humidity_3": "Humidity (%)", 
                   "time": "Time"}, inplace=True)

df_melted = pd.melt(df, id_vars=["Time"], var_name="Sensor", value_name="Value")
g = sns.FacetGrid(df_melted, col="Sensor", col_wrap=1, sharex=True, sharey=False, height=2.5, aspect=5)
g.map(sns.lineplot, "Time", "Value")
g.set_titles("")
g.set_xlabels("Time")
for ax, name in zip(g.axes.flat, g.col_names):
    ax.set_ylabel(name)
    ax.grid(True)
for ax in g.axes.flat:
    for label in ax.get_xticklabels():
        label.set_rotation(30)
        label.set_ha("right")
palette = sns.color_palette("tab10", n_colors=len(g.col_names))
for ax, color in zip(g.axes.flat, palette):
    lines = ax.get_lines()
    for line in lines:
        line.set_color(color)
plt.tight_layout()
plt.show()